# Fair LLM Evaluation

This notebook evaluates the RAG/LLM pipeline on the same `test.csv` courses used by the supervised models. The label universe is fitted from `train.csv`, and metrics are exported with the same multi-label protocol used elsewhere in the project.

In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from pipeline_utils import (
    DEFAULT_K_VALUES,
    compute_ranked_label_metrics,
    export_experiment_artifacts,
    load_split_csv,
    prepare_multilabel_targets,
)

load_dotenv()

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
RAW_PATH = "dataset.csv"
TOP_K_MAX = max(DEFAULT_K_VALUES)
SEED = 42

NOTEBOOK_NAME = "llm_open.ipynb"
PROVIDER = "ollama"
RUNTIME = "local_ollama"
HARDWARE_TAG = "rtx6000_ada_48gb"
HARDWARE_DESCRIPTION = "NVIDIA RTX 6000 Ada 48GB"
METHOD_TAG = "ollama_gemma3_27b_nomic_embed_rag"
#METHOD_TAG = "ollama_deepseek_r1_70b_nomic_embed_rag"
RESULTS_DIR = "results/llm_open"
VECTORSTORE_DIR = Path("results/vectorstores") / METHOD_TAG

In [10]:
train_df = load_split_csv(TRAIN_PATH)
test_df = load_split_csv(TEST_PATH)
y_train, y_test, classes, mlb = prepare_multilabel_targets(train_df, test_df)

raw_df = pd.read_csv(RAW_PATH)
valid_label_set = set(classes)

df_courses = test_df[["courseId", "courseName", "courseDescription", "combinedText", "comp_name_filtered"]].copy()
df_courses = df_courses.reset_index(drop=True)

df_competencies = (
    raw_df[raw_df["comp_name"].isin(valid_label_set)]
    [["comp_id", "comp_name", "comp_description", "cat_id", "cat_name"]]
    .dropna(subset=["comp_id", "comp_name"])
    .sort_values(["comp_name", "comp_id"])
    .drop_duplicates(subset=["comp_name"])
    .reset_index(drop=True)
)
df_competencies["comp_description"] = df_competencies["comp_description"].fillna("No description available.")

missing_labels = sorted(valid_label_set - set(df_competencies["comp_name"]))
if missing_labels:
    raise ValueError(f"Missing RAG candidates for labels: {missing_labels}")
if len(df_competencies) != len(classes):
    raise ValueError(f"RAG candidate count ({len(df_competencies)}) must match ML label count ({len(classes)}).")

comp_id_to_name = dict(zip(df_competencies["comp_id"].astype(str), df_competencies["comp_name"].astype(str)))
valid_comp_ids = set(comp_id_to_name)

print(f"Train courses: {len(train_df)}")
print(f"Test courses: {len(df_courses)}")
print(f"Valid labels: {len(classes)}")
print(f"Candidate competencies in vector store: {len(df_competencies)}")
df_courses.head()


Train courses: 270
Test courses: 68
Valid labels: 53
Candidate competencies in vector store: 53


,courseId,courseName,courseDescription,combinedText,comp_name_filtered
0,041220d0-21c3-4f7b-8e27-ca87b810f2f0,Desenvolvimento Humano e Educação,O curso Desenvolvimento Humano e Educação é um...,Desenvolvimento Humano e Educação O curso Dese...,"[Ensino, Orientação]"
1,04c73a54-de08-41bd-ae53-23f54a207fdf,Direitos Humanos: Uma Declaração Universal,Objetivo:\nPromover a compreensão e aplicação ...,Direitos Humanos: Uma Declaração Universal Obj...,"[Ensino, Orientação]"
2,0b90941f-f127-42d8-b68f-b0dd26bc3f4b,Introdução à Educação de Surdos,Introdução à Educação de Surdos é um curso que...,Introdução à Educação de Surdos Introdução à E...,"[Acessibilidade e inclusão - Docente, Diferenc..."
3,122a7dfa-0395-452d-b625-481221db8953,Como elaborar editais e construir pareceres,Este curso tem como objetivo promover a formaç...,Como elaborar editais e construir pareceres Es...,"[Acessibilidade e inclusão - Docente, Promoção..."
4,17d726c1-89a4-4bd1-a163-7879ffbfe5e2,Temos que dar aulas remotas... E agora?,As aulas presenciais estão suspensas e temos q...,Temos que dar aulas remotas... E agora? As aul...,"[Criação de conteúdo digital, Criação e modifi..."


In [ ]:
from langchain_core.messages import AIMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings

MODEL = "gemma3:27b"
#MODEL = "deepseek-r1:70b"
EMBEDDING_MODEL = "nomic-embed-text"

llm = ChatOllama(model=MODEL, temperature=0.0)
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)


In [12]:
texts = []
metadatas = []

for _, row in df_competencies.iterrows():
    text = (
        f"ID: {row['comp_id']}\n"
        f"Name: {row['comp_name']}\n"
        f"Description: {row['comp_description']}"
    )
    texts.append(text)
    metadatas.append({
        "comp_id": str(row["comp_id"]),
        "comp_name": str(row["comp_name"]),
    })

vectorstore_manifest = {
    "provider": PROVIDER,
    "embedding_model": EMBEDDING_MODEL,
    "retriever_top_k": TOP_K_MAX,
    "competencies": [
        {"text": text, "metadata": metadata}
        for text, metadata in zip(texts, metadatas)
    ],
}
vectorstore_manifest_hash = hashlib.sha256(
    json.dumps(vectorstore_manifest, ensure_ascii=False, sort_keys=True).encode("utf-8")
).hexdigest()
manifest_path = VECTORSTORE_DIR / "manifest.json"

vectorstore_reused = False
vectorstore_build_seconds = 0.0
vectorstore_load_seconds = 0.0

if manifest_path.exists():
    stored_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    vectorstore_reused = stored_manifest.get("manifest_hash") == vectorstore_manifest_hash

if vectorstore_reused:
    vectorstore_start = time.perf_counter()
    vectorstore = FAISS.load_local(
        str(VECTORSTORE_DIR),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    vectorstore_load_seconds = time.perf_counter() - vectorstore_start
else:
    vectorstore_start = time.perf_counter()
    vectorstore = FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)
    vectorstore_build_seconds = time.perf_counter() - vectorstore_start
    VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)
    vectorstore.save_local(str(VECTORSTORE_DIR))
    manifest_payload = dict(vectorstore_manifest)
    manifest_payload["manifest_hash"] = vectorstore_manifest_hash
    manifest_payload["index_path"] = str(VECTORSTORE_DIR)
    manifest_path.write_text(json.dumps(manifest_payload, ensure_ascii=False, indent=2), encoding="utf-8")

setup_seconds = vectorstore_load_seconds if vectorstore_reused else vectorstore_build_seconds
setup_operation = "vectorstore_load" if vectorstore_reused else "vectorstore_build"
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K_MAX})

print(f"Vector store {setup_operation} completed in {setup_seconds:.2f} seconds")


Vector store vectorstore_build completed in 1.01 seconds


In [13]:
class CourseLabel(BaseModel):
    course_id: str = Field(..., description="Course ID (courseId)")
    selected_competencies: List[str] = Field(..., description="Ranked list of selected competency IDs (comp_id)")

parser = JsonOutputParser(pydantic_object=CourseLabel)

SYSTEM_PROMPT_TEMPLATE = """
You are an expert in mapping courses to competencies.

All course titles, descriptions, and competencies are written in Brazilian Portuguese.
DO NOT TRANSLATE, REWRITE, OR MODIFY ANY OF THESE TEXTS. USE THEM ONLY AS EVIDENCE.

Task:
Given a course description and a list of candidate competencies, rank up to {top_k} competencies
that are explicitly related to the course content.

Instructions:
- CONSIDER ONLY the candidate competencies provided.
- USE ONLY the course title and description as evidence.
- Evaluate each competency based on its name and description (in Portuguese).
- Select a competency ONLY IF THERE IS A CLEAR AND EXPLICIT CONNECTION to the course.
- Return the selected competencies in descending relevance order.
- RETURN AT MOST {top_k} COMPETENCIES.
- If none apply, RETURN AN EMPTY LIST.

Rules:
- DO NOT REWRITE, TRANSLATE, OR MODIFY ANY COMPETENCY TEXT.
- DO NOT HALLUCINATE NEW COMPETENCIES.
- BASE ALL DECISIONS STRICTLY ON THE PROVIDED TEXT.
- RETURN ONLY THE JSON OBJECT DEFINED BY THE SCHEMA.
"""

SYSTEM_PROMPT = SYSTEM_PROMPT_TEMPLATE.format(top_k=TOP_K_MAX)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        (
            "human",
            """Course (ID: {course_id})

Title: {course_title}
Description: {course_description}

Candidate competencies (ID - Name: Description):
{candidate_competencies}

{format_instructions}
""",
        ),
    ]
)


In [14]:
chain = prompt | llm


def clean_llm_output_to_json(text: str) -> str:
    s = str(text)
    if "<think>" in s and "</think>" in s:
        s = s.split("</think>", 1)[1]
    s = s.strip()
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"Could not find JSON in the model output:\n{s}")
    return s[start : end + 1].strip()


In [15]:
def build_candidate_text(docs):
    lines = []
    for d in docs:
        comp_id = d.metadata["comp_id"]
        comp_name = d.metadata["comp_name"]
        lines.append(f"{comp_id} - {comp_name}: {d.page_content}")
    return "\n".join(lines)


def normalize_selected_competencies(selected):
    labels = []
    raw_ids = []
    for item in selected or []:
        item = str(item).strip()
        if item in comp_id_to_name:
            label = comp_id_to_name[item]
            raw_ids.append(item)
        elif item in valid_label_set:
            label = item
            raw_ids.append(None)
        else:
            continue
        if label in valid_label_set and label not in labels:
            labels.append(label)
    return raw_ids, labels


def label_course(course_row):
    course_id = str(course_row["courseId"])
    course_name = str(course_row["courseName"])
    course_description = str(course_row["courseDescription"])
    course_text = f"{course_name}: {course_description}"

    retrieval_start = time.perf_counter()
    candidates = retriever.invoke(course_text)
    retrieval_seconds = time.perf_counter() - retrieval_start

    llm_start = time.perf_counter()
    raw = chain.invoke(
        {
            "course_id": course_id,
            "course_title": course_name,
            "course_description": course_description,
            "candidate_competencies": build_candidate_text(candidates),
            "format_instructions": parser.get_format_instructions(),
        }
    )
    llm_seconds = time.perf_counter() - llm_start

    text = raw.content if isinstance(raw, AIMessage) else str(raw)
    result = parser.parse(clean_llm_output_to_json(text))
    raw_ids, selected_labels = normalize_selected_competencies(result.get("selected_competencies", []))
    return {
        "courseId": course_id,
        "selected_competencies": raw_ids,
        "selected_labels": selected_labels,
        "retrieved_competencies": [d.metadata["comp_id"] for d in candidates],
        "retrieval_seconds": retrieval_seconds,
        "llm_seconds": llm_seconds,
    }


In [16]:
raw_predictions_path = Path(RESULTS_DIR) / f"{METHOD_TAG}_raw_predictions.csv"
errors_path = Path(RESULTS_DIR) / f"{METHOD_TAG}_errors.csv"
raw_predictions_path.parent.mkdir(parents=True, exist_ok=True)

LIST_COLUMNS = ["selected_competencies", "selected_labels", "retrieved_competencies"]


def serialize_checkpoint(df):
    output = df.copy()
    for col in LIST_COLUMNS:
        if col in output.columns:
            output[col] = output[col].apply(lambda value: json.dumps(value if isinstance(value, list) else [], ensure_ascii=False))
    return output


def load_prediction_checkpoint(path):
    if not path.exists():
        return pd.DataFrame()
    checkpoint = pd.read_csv(path)
    for col in LIST_COLUMNS:
        if col in checkpoint.columns:
            checkpoint[col] = checkpoint[col].apply(lambda value: json.loads(value) if isinstance(value, str) else [])
    checkpoint["courseId"] = checkpoint["courseId"].astype(str)
    return checkpoint


df_checkpoint = load_prediction_checkpoint(raw_predictions_path)
results_by_course = {
    str(row["courseId"]): row.to_dict()
    for _, row in df_checkpoint.iterrows()
}
errors = pd.read_csv(errors_path).to_dict("records") if errors_path.exists() and errors_path.stat().st_size > 0 else []
processed_course_ids = set(results_by_course)
pending_courses = df_courses[~df_courses["courseId"].astype(str).isin(processed_course_ids)]

print(f"Resuming from {len(processed_course_ids)} saved predictions; {len(pending_courses)} courses pending.")

inference_start = time.perf_counter()
for _, row in tqdm(pending_courses.iterrows(), total=len(pending_courses), desc=f"{METHOD_TAG} inference"):
    course_id = str(row.get("courseId"))
    try:
        result = label_course(row)
    except Exception as exc:
        errors.append({"courseId": course_id, "error": str(exc)})
        result = {
            "courseId": course_id,
            "selected_competencies": [],
            "selected_labels": [],
            "retrieved_competencies": [],
            "retrieval_seconds": 0.0,
            "llm_seconds": 0.0,
        }
    results_by_course[course_id] = result
    ordered_results = [
        results_by_course[str(course_id)]
        for course_id in df_courses["courseId"].astype(str)
        if str(course_id) in results_by_course
    ]
    serialize_checkpoint(pd.DataFrame(ordered_results)).to_csv(raw_predictions_path, index=False)
    pd.DataFrame(errors, columns=["courseId", "error"]).to_csv(errors_path, index=False)
inference_seconds = time.perf_counter() - inference_start

missing_course_ids = [
    str(course_id)
    for course_id in df_courses["courseId"].astype(str)
    if str(course_id) not in results_by_course
]
if missing_course_ids:
    raise RuntimeError(f"Missing predictions for {len(missing_course_ids)} courses. Re-run this cell to resume.")

df_pred = pd.DataFrame([
    results_by_course[str(course_id)]
    for course_id in df_courses["courseId"].astype(str)
])
df_errors = pd.DataFrame(errors, columns=["courseId", "error"])

ranked_predictions = df_pred["selected_labels"].tolist()
metrics_df, predictions_df = compute_ranked_label_metrics(
    y_test,
    ranked_predictions,
    labels=classes,
    method=METHOD_TAG,
    k_values=DEFAULT_K_VALUES,
)

timing_rows = [
    {
        "method": METHOD_TAG,
        "setup_seconds": setup_seconds,
        "setup_operation": setup_operation,
        "fit_or_setup_seconds": setup_seconds,
        "vectorstore_reused": vectorstore_reused,
        "vectorstore_build_seconds": vectorstore_build_seconds,
        "vectorstore_load_seconds": vectorstore_load_seconds,
        "inference_seconds": inference_seconds,
        "inference_seconds_per_sample": inference_seconds / max(1, len(df_courses)),
        "total_runtime_seconds": setup_seconds + inference_seconds,
        "total_runtime_seconds_per_sample": (setup_seconds + inference_seconds) / max(1, len(df_courses)),
        "retrieval_seconds": float(df_pred["retrieval_seconds"].sum()),
        "retrieval_seconds_per_sample": float(df_pred["retrieval_seconds"].mean()),
        "llm_seconds": float(df_pred["llm_seconds"].sum()),
        "llm_seconds_per_sample": float(df_pred["llm_seconds"].mean()),
    }
]

paths = export_experiment_artifacts(
    results_dir=RESULTS_DIR,
    method=METHOD_TAG,
    metrics_df=metrics_df,
    predictions_df=predictions_df,
    timing_rows=timing_rows,
    config={
        "notebook": NOTEBOOK_NAME,
        "method": METHOD_TAG,
        "provider": PROVIDER,
        "runtime": RUNTIME,
        "hardware_tag": HARDWARE_TAG,
        "hardware": HARDWARE_DESCRIPTION,
        "model": MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "train_path": TRAIN_PATH,
        "test_path": TEST_PATH,
        "raw_path": RAW_PATH,
        "seed": SEED,
        "k_values": DEFAULT_K_VALUES,
        "retriever_top_k": TOP_K_MAX,
        "faiss": {
            "index": "FAISS",
            "index_path": str(VECTORSTORE_DIR),
            "manifest_hash": vectorstore_manifest_hash,
            "reused": vectorstore_reused,
            "candidate_count": int(len(df_competencies)),
        },
        "labels": list(classes),
        "n_train": int(y_train.shape[0]),
        "n_test": int(y_test.shape[0]),
        "n_errors": int(len(df_errors)),
    },
)

serialize_checkpoint(df_pred).to_csv(raw_predictions_path, index=False)
df_errors.to_csv(errors_path, index=False)

print(metrics_df)
print("Exported artifacts:", paths)
print(f"Raw predictions: {raw_predictions_path}")
print(f"Errors: {errors_path} ({len(df_errors)} rows)")


Resuming from 0 saved predictions; 68 courses pending.


ollama_gemma3_27b_nomic_embed_rag inference: 100%|██████████| 68/68 [12:26<00:00, 10.97s/it]  

                              method   k  micro_f1  macro_f1  hamming_loss  \
0  ollama_gemma3_27b_nomic_embed_rag   1  0.190871  0.109589      0.054107   
1  ollama_gemma3_27b_nomic_embed_rag   3  0.240000  0.139215      0.079079   
2  ollama_gemma3_27b_nomic_embed_rag   5  0.214737  0.140976      0.103496   
3  ollama_gemma3_27b_nomic_embed_rag   7  0.224000  0.142180      0.107658   
4  ollama_gemma3_27b_nomic_embed_rag  10  0.225743  0.141867      0.108491   

   subset_accuracy  precision_at_k  recall_at_k  partial_hit_at_k  lrap  \
0         0.044118        0.338235     0.145700          0.338235   NaN   
1         0.000000        0.220588     0.256840          0.470588   NaN   
2         0.000000        0.150000     0.290530          0.500000   NaN   
3         0.000000        0.117647     0.322883          0.529412   NaN   
4         0.000000        0.083824     0.325334          0.529412   NaN   

   coverage_error  n_samples  n_labels  
0             NaN         68        53 